# Validation (a): slab waveguide TE modes vs the analytic dispersion relation

**Claim checked.** The analytic TE solver (`analytic_te_modes`, which solves the
transcendental dispersion relation) and an independent 1-D finite-difference
eigen-solver (`finite_difference_te_modes`) agree on every guided mode, and
the finite-difference error falls at second order as the grid is refined.

**Tolerance.** At 400 points/µm, |Δn_eff| < 1×10⁻⁵ for every mode. The observed
convergence order is between 1.8 and 2.2. The mode count equals ⌈2V/π⌉.

**Fidelity.** The analytic solver is *analytic*. The finite-difference solver is
*full-wave* (exact Maxwell for TE in 1-D, discretized). Neither is a measurement.

References: K. Okamoto, *Fundamentals of Optical Waveguides*, 2nd ed.
(Academic Press, 2006), ch. 2; B. E. A. Saleh and M. C. Teich, *Fundamentals
of Photonics*, 3rd ed. (Wiley, 2019).

In [ ]:
import math
from awen_physics.slab import SlabWaveguide, analytic_te_modes, finite_difference_te_modes

# Silicon-nitride-like core in silica at 1550 nm (illustrative indices).
guide = SlabWaveguide(core_index=2.0, cladding_index=1.45, thickness_um=1.0)
wavelength = 1.55
analytic = analytic_te_modes(guide, wavelength)
print("V =", round(guide.v_number(wavelength), 4), "| TE modes:", guide.te_mode_count(wavelength))
print("analytic n_eff:", [round(n, 8) for n in analytic.effective_indices])
assert len(analytic.effective_indices) == guide.te_mode_count(wavelength)

In [ ]:
errors = {}
for points in (50, 100, 200, 400):
    numeric = finite_difference_te_modes(guide, wavelength, points_per_um=points)
    assert len(numeric.effective_indices) == len(analytic.effective_indices)
    errors[points] = [abs(a - b) for a, b in zip(numeric.effective_indices, analytic.effective_indices)]
    print(points, "pts/µm  |Δn_eff| =", ["%.2e" % e for e in errors[points]])

orders = [math.log2(errors[p][m] / errors[2 * p][m]) for p in (100, 200) for m in range(len(analytic.effective_indices))]
print("observed convergence orders:", [round(o, 3) for o in orders])
assert max(errors[400]) < 1e-5, errors[400]
assert all(1.8 < o < 2.2 for o in orders), orders

In [ ]:
# Dispersion: n_eff of the fundamental mode across wavelength, both methods.
for wl in (1.30, 1.55, 2.00):
    a = analytic_te_modes(guide, wl).effective_indices[0]
    f = finite_difference_te_modes(guide, wl, points_per_um=400).effective_indices[0]
    print(f"{wl:.2f} µm  analytic {a:.7f}  numeric {f:.7f}  Δ {abs(a - f):.1e}")
    assert abs(a - f) < 1e-5

**Result.** The two methods agree within the stated tolerance at every
wavelength. The finite-difference error converges at second order, as expected
for central differences with the interfaces placed midway between grid points.